# From equilibrium state to device prediction

This notebook closes the LA1-to-LA2 instructional loop: chemical-potential built-in voltage, generalized Einstein diffusion, incomplete-ionization depletion charge, and uncertainty ranking.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import Boltzmann as k_B, elementary_charge as q, epsilon_0
from scipy.integrate import quad
from scipy.special import expit, gamma

ROOT=Path.cwd()
while not (ROOT/'figures'/'src').exists():
    if ROOT.parent==ROOT: raise RuntimeError('Could not locate lecture-notes/figures/src')
    ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'figures'/'src'))

# Direct normalized Fermi-integral calculation.
def direct_F(order,eta):
    value,_=quad(lambda x:x**order*expit(eta-x),0,np.inf,epsabs=1e-11,epsrel=1e-10)
    return value/gamma(order+1)
eta=np.array([-8.,0.,5.])
factor=np.array([direct_F(.5,e)/direct_F(-.5,e) for e in eta])
print('generalized Einstein factors:',dict(zip(eta,factor)))

generalized Einstein factors: {np.float64(-8.0): np.float64(1.0001185887725408), np.float64(0.0): np.float64(1.2649177394370985), np.float64(5.0): np.float64(3.576325580781004)}


In [2]:
# Direct LA1-state transfer: both chemical potentials use one energy reference.
T=300.; VT=k_B*T/q; ni=1e16; n_n=1e22; p_p=1e23
mu_n=VT*np.log(n_n/ni)
mu_p=-VT*np.log(p_p/ni)
Vbi_mu=mu_n-mu_p
Vbi_carrier=VT*np.log(n_n*p_p/ni**2)
print(f'Vbi from chemical potentials = {Vbi_mu:.9f} V')
print(f'Vbi from carrier states      = {Vbi_carrier:.9f} V')
assert abs(Vbi_mu-Vbi_carrier)<1e-12
for shift in (-2.,0.,3.): assert abs((mu_n+shift)-(mu_p+shift)-Vbi_mu)<1e-12

# Incomplete ionization changes fixed depletion charge.
NA,ND=5e22,1e22;eps=11.7*epsilon_0
for fraction in (1.0,0.5,0.2):
    W=np.sqrt(2*eps*Vbi_mu/q*(1/(fraction*NA)+1/(fraction*ND)))
    print(f'ionized fraction={fraction:.1f}: W={W*1e6:.3f} µm')

Vbi from chemical potentials = 0.773843581 V
Vbi from carrier states      = 0.773843581 V
ionized fraction=1.0: W=0.347 µm
ionized fraction=0.5: W=0.490 µm
ionized fraction=0.2: W=0.775 µm


In [3]:
from semphy.device_prediction import generalized_einstein_factor,built_in_from_carrier_states,built_in_from_chemical_potentials,depletion_width_from_ionized_dopants

np.testing.assert_allclose(generalized_einstein_factor(eta),factor,rtol=1e-12)
np.testing.assert_allclose(built_in_from_carrier_states(n_n,p_p,ni,T),Vbi_mu)
np.testing.assert_allclose(built_in_from_chemical_potentials(mu_n,mu_p),Vbi_mu)
W,xp,xn=depletion_width_from_ionized_dopants(Vbi_mu,0.,NA,ND,eps)
print(f'depletion charge-neutrality error = {abs(NA*xp-ND*xn)/(ND*xn):.3e}')

depletion charge-neutrality error = 0.000e+00


In [4]:
from semphy.device_prediction import integrated_prediction_figure
integrated_prediction_figure()
print('wrote figures/device-prediction-integration.pdf')

wrote figures/device-prediction-integration.pdf
